## 1. Install Dependencies

In [ ]:
%pip install -U torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128
%pip install -U pnnx==20250725 ncnn ultralytics==8.3.236
%pip install roboflow

## 2. Download and prepare dataset

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="Mt0m4dCKTTlNf29OV3zm")
project = rf.workspace("dylans-workspace-3init").project("my-first-project-sbk3a")
version = project.version(1)
dataset = version.download("yolov11") # This downloads the images AND the data.yaml

### 2.1 Patch dataset with missing labels

In [ ]:
from pathlib import Path

dataset_root = Path("My-First-Project-1")
OLD_BLOCK = """nc: 1
names: ['sports_ball']"""
NEW_BLOCK = """names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
  8: boat
  9: traffic light
  10: fire hydrant
  11: stop sign
  12: parking meter
  13: bench
  14: bird
  15: cat
  16: dog
  17: horse
  18: sheep
  19: cow
  20: elephant
  21: bear
  22: zebra
  23: giraffe
  24: backpack
  25: umbrella
  26: handbag
  27: tie
  28: suitcase
  29: frisbee
  30: skis
  31: snowboard
  32: sports ball
  33: kite
  34: baseball bat
  35: baseball glove
  36: skateboard
  37: surfboard
  38: tennis racket
  39: bottle
  40: wine glass
  41: cup
  42: fork
  43: knife
  44: spoon
  45: bowl
  46: banana
  47: apple
  48: sandwich
  49: orange
  50: broccoli
  51: carrot
  52: hot dog
  53: pizza
  54: donut
  55: cake
  56: chair
  57: couch
  58: potted plant
  59: bed
  60: dining table
  61: toilet
  62: tv
  63: laptop
  64: mouse
  65: remote
  66: keyboard
  67: cell phone
  68: microwave
  69: oven
  70: toaster
  71: sink
  72: refrigerator
  73: book
  74: clock
  75: vase
  76: scissors
  77: teddy bear
  78: hair drier
  79: toothbrush"""
def rewrite_data_yaml( dry_run: bool = False):
    path = dataset_root / "data.yaml"
    text = path.read_text(encoding="utf-8")

    if OLD_BLOCK not in text:
        raise ValueError("Target block not found. Nothing replaced.")

    updated = text.replace(OLD_BLOCK, NEW_BLOCK, 1)

    if dry_run:
        print("Dry run: replacement would be applied.")
        return

    path.write_text(updated, encoding="utf-8")
    print(f"Updated: {path}")

def rewrite_label_file(path: Path, target_class: int, dry_run: bool = False):
    changed_lines = 0
    original = path.read_text(encoding="utf-8").splitlines(keepends=True)
    updated = []

    for line in original:
        stripped = line.strip()

        # Keep blank lines unchanged
        if not stripped:
            updated.append(line)
            continue

        # Split into: first token (class id) + rest (coords/polygon points)
        parts = stripped.split(maxsplit=1)
        first = parts[0]
        rest = parts[1] if len(parts) > 1 else ""

        # Only rewrite lines that start with a numeric class id
        try:
            float(first)
        except ValueError:
            updated.append(line)
            continue

        new_line = f"{target_class} {rest}".rstrip()
        if line.endswith("\n"):
            new_line += "\n"

        if new_line != line:
            changed_lines += 1
        updated.append(new_line)

    if changed_lines and not dry_run:
        path.write_text("".join(updated), encoding="utf-8")

    return changed_lines

def change_label_classes(dry_run: bool = True):
    target_class = 32
    label_dirs = [
        dataset_root / "train" / "labels",
        dataset_root / "valid" / "labels",
        dataset_root / "test" / "labels",
    ]

    total_files = 0
    touched_files = 0
    total_lines_changed = 0

    for label_dir in label_dirs:
        if not label_dir.exists():
            print(f"Skipping missing directory: {label_dir}")
            continue

        for txt_file in label_dir.rglob("*.txt"):
            total_files += 1
            changed = rewrite_label_file(txt_file, target_class, dry_run=dry_run)
            if changed:
                touched_files += 1
                total_lines_changed += changed
                print(f"Updated {txt_file} ({changed} line(s))")

    mode = "DRY RUN" if dry_run else "WRITE"
    print(f"\n[{mode}] Scanned files: {total_files}")
    print(f"[{mode}] Files changed: {touched_files}")
    print(f"[{mode}] Label lines changed: {total_lines_changed}")


dry_run = False  # Set to False to actually write changes
rewrite_data_yaml(dry_run=dry_run)
change_label_classes(dry_run=dry_run)

# 3. Train

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from ultralytics import YOLO, settings
settings.reset() # reset the output dirs if different envs messed with it


# 1. Load a pretrained YOLO11n model
model = YOLO('yolo11n.pt')

# 2. Train the model
# 'data.yaml' contains paths to your orange ball images and class names
model.train(data=f"{dataset.location}/data.yaml", epochs=100, imgsz=640,batch=32)

## 3. Export YOLO11 NCNN

In [2]:
train_dir = "runs/detect/train/weights"

In [5]:
from pathlib import Path
repo_dir = Path.cwd().resolve()
%cd {train_dir}

!yolo export model=best.pt format=torchscript
!pnnx best.torchscript

import os
def patch_yolo_script(filename: str):
    if not os.path.exists(filename):
        print(f"File {filename} not found, skipping.")
        return

    with open(filename, 'r') as f:
        script = f.read()

    chunk1 = """        v_235 = v_204.view(1, 144, 6400)
        v_236 = v_219.view(1, 144, 1600)
        v_237 = v_234.view(1, 144, 400)
        v_238 = torch.cat((v_235, v_236, v_237), dim=2)"""
    replace1 = """        v_235 = v_204.view(1, 144, -1).transpose(1, 2)
        v_236 = v_219.view(1, 144, -1).transpose(1, 2)
        v_237 = v_234.view(1, 144, -1).transpose(1, 2)
        v_238 = torch.cat((v_235, v_236, v_237), dim=1)
        return v_238"""
    chunk2 = """        v_95 = self.model_10_m_0_attn_qkv_conv(v_94)
        v_96 = v_95.view(1, 2, 128, 400)
        v_97, v_98, v_99 = torch.split(tensor=v_96, dim=2, split_size_or_sections=(32,32,64))
        v_100 = torch.transpose(input=v_97, dim0=-2, dim1=-1)
        v_101 = torch.matmul(input=v_100, other=v_98)
        v_102 = (v_101 * 0.176777)
        v_103 = F.softmax(input=v_102, dim=-1)
        v_104 = torch.transpose(input=v_103, dim0=-2, dim1=-1)
        v_105 = torch.matmul(input=v_99, other=v_104)
        v_106 = v_105.view(1, 128, 20, 20)
        v_107 = v_99.reshape(1, 128, 20, 20)
        v_108 = self.model_10_m_0_attn_pe_conv(v_107)
        v_109 = (v_106 + v_108)
        v_110 = self.model_10_m_0_attn_proj_conv(v_109)"""
    replace2 = """        v_95 = self.model_10_m_0_attn_qkv_conv(v_94)
        v_96 = v_95.view(1, 2, 128, -1) # <--- This line, note this v_95
        v_97, v_98, v_99 = torch.split(tensor=v_96, dim=2, split_size_or_sections=(32,32,64))
        v_100 = torch.transpose(input=v_97, dim0=-2, dim1=-1)
        v_101 = torch.matmul(input=v_100, other=v_98)
        v_102 = (v_101 * 0.176777)
        v_103 = F.softmax(input=v_102, dim=-1)
        v_104 = torch.transpose(input=v_103, dim0=-2, dim1=-1)
        v_105 = torch.matmul(input=v_99, other=v_104)
        v_106 = v_105.view(1, 128, v_95.size(2), v_95.size(3)) # <--- This line
        v_107 = v_99.reshape(1, 128, v_95.size(2), v_95.size(3)) # <--- This line
        v_108 = self.model_10_m_0_attn_pe_conv(v_107)
        v_109 = (v_106 + v_108)
        v_110 = self.model_10_m_0_attn_proj_conv(v_109)"""
    if chunk1 not in script:
        raise ValueError("Target code block not found. No changes applied.")
    script = script.replace(chunk1, replace1)

    if chunk2 not in script:
        raise ValueError("Target code block not found. No changes applied.")
    script = script.replace(chunk2, replace2)


    with open(filename, 'w') as f:
        f.write(script)
    print(f"Patched {filename} successfully.")


patch_yolo_script('best_pnnx.py')

import best_pnnx
best_pnnx.export_torchscript()
!pnnx best_pnnx.py.pt inputshape=[1,3,640,640] inputshape2=[1,3,320,320]
%cd {repo_dir}

[WinError 3] The system cannot find the path specified: 'runs/detect/train/weights'
c:\Users\thesp\Documents\Projects\Unibots\BARF\runs\detect\train\weights
Ultralytics 8.3.236  Python-3.13.1 torch-2.8.0+cu128 CPU (AMD Ryzen 7 5800H with Radeon Graphics)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs

PyTorch: starting from 'best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (5.3 MB)

TorchScript: starting export with torch 2.8.0+cu128...
TorchScript: export success  2.1s, saved as 'best.torchscript' (10.5 MB)

Export complete (2.5s)
Results saved to C:\Users\thesp\Documents\Projects\Unibots\BARF\runs\detect\train\weights
Predict:         yolo predict task=detect model=best.torchscript imgsz=640  
Validate:        yolo val task=detect model=best.torchscript imgsz=640 data=c:\Users\thesp\Documents\Projects\Unibots\BARF\My-First-Project-1/data.yaml  
Visualize:       https://netron.app
 Learn more at https://docs.ultralyt

pnnxparam = best.pnnx.param
pnnxbin = best.pnnx.bin
pnnxpy = best_pnnx.py
pnnxonnx = best.pnnx.onnx
ncnnparam = best.ncnn.param
ncnnbin = best.ncnn.bin
ncnnpy = best_ncnn.py
fp16 = 1
optlevel = 2
device = cpu
inputshape = 
inputshape2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.Attention
inline module = ultralytics.nn.modules.block.Bottleneck
inline module = ultralytics.nn.modules.block.C2PSA
inline module = ultralytics.nn.modules.block.C3k
inline module = ultralytics.nn.modules.block.C3k2
inline module = ultralytics.nn.modules.block.DFL
inline module = ultralytics.nn.modules.block.PSABlock
inline module = ultralytics.nn.modules.block.SPPF
inline module = ultralytics.nn.modules.conv.Concat
inline module = ultralytics.nn.modules.conv.Conv
inline module = ultralytics.nn.modules.conv.DWConv
inline module = ultralytics.nn.mod

Patched best_pnnx.py successfully.
C:\Users\thesp\Documents\Projects\Unibots\BARF\runs\detect\train\weights


pnnxparam = best_pnnx.py.pnnx.param
pnnxbin = best_pnnx.py.pnnx.bin
pnnxpy = best_pnnx.py_pnnx.py
pnnxonnx = best_pnnx.py.pnnx.onnx
ncnnparam = best_pnnx.py.ncnn.param
ncnnbin = best_pnnx.py.ncnn.bin
ncnnpy = best_pnnx.py_ncnn.py
fp16 = 1
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = [1,3,320,320]f32
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0

----------------

assign dynamic shape info
############# pass_level1
############# pass_level2
############# pass_level3
############# pass_level4
############# pass_level5
############# pass_ncnn
convert reshape expression [1,128,size(@0,2),size(@0,3)] => 1w,1h,128
convert reshape expression [1,128,size(@0,2),size(@0,3)] => 1w,1h,128


### 3.1 Copy the exported model to app assets

In [10]:
import shutil

assets_dir = repo_dir / "app" / "src" / "main" / "assets"
renames = {
    "best_pnnx.py.ncnn.bin": "yolo11n.ncnn.bin",
    "best_pnnx.py.ncnn.param": "yolo11n.ncnn.param",
}
for src, dst in renames.items():
    shutil.copy((Path(train_dir)/src).resolve(), (assets_dir / dst).resolve())
    print(f"Copied {src} -> {assets_dir / dst}")

Copied best_pnnx.py.ncnn.bin -> C:\Users\thesp\Documents\Projects\Unibots\BARF\app\src\main\assets\yolo11n.ncnn.bin
Copied best_pnnx.py.ncnn.param -> C:\Users\thesp\Documents\Projects\Unibots\BARF\app\src\main\assets\yolo11n.ncnn.param
